# Is This AI Slop? (ITAIS) — Colab Run All

**Runtime → Change runtime type → T4 GPU** (CPU works for smoke). Then **Runtime → Run all**.

Smoke trains a calibrated `matches_ai_pile` head on weak labels. It does not claim authorship.
Gemma-3-4B residual distillation is optional (`FULL = True` + Hugging Face token with Gemma access).
If Gemma fails, Qwen2.5-0.5B is the fallback. If there is no GPU, that cell prints `skipped` and the demo still runs.


In [ ]:
# Config
SMOKE = True          # False = more HC3 rows + try Gemma
FULL = False          # True requires HF_TOKEN with Gemma license accepted
MOUNT_DRIVE = True
N_DOCS = 400 if SMOKE else 4000

import os, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    drive = None
    userdata = None

if IN_COLAB and MOUNT_DRIVE:
    try:
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/isthisaislop")
    except Exception as exc:
        print("Drive mount failed, using /content:", exc)
        ROOT = Path("/content/isthisaislop")
elif IN_COLAB:
    ROOT = Path("/content/isthisaislop")
else:
    ROOT = Path(".").resolve()
    if not (ROOT / "src" / "slopdet").exists():
        ROOT = Path("/home/vstaln/slop-detector")

ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("ROOT", ROOT)

if IN_COLAB:
    try:
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            print("HF_TOKEN loaded from Colab secrets")
    except Exception:
        pass
if FULL:
    os.environ["FULL"] = "1"

import subprocess, sys
pkgs = ["pyyaml", "regex", "jsonschema", "scikit-learn", "datasets", "numpy>=1.26"]
if IN_COLAB:
    pkgs += ["transformers", "accelerate", "bitsandbytes", "safetensors"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ok")


In [ ]:
# Write the package onto disk (self-contained; no git clone required)
import json
from pathlib import Path
FILES = json.loads('{"pyproject.toml": "[build-system]\\nrequires = [\\"setuptools>=68\\", \\"wheel\\"]\\nbuild-backend = \\"setuptools.build_meta\\"\\n\\n[project]\\nname = \\"slopdet\\"\\nversion = \\"0.1.0\\"\\ndescription = \\"Is This AI Slop? (ITAIS) \\u2014 checkable style hits plus an AI-pile resemblance score.\\"\\nreadme = \\"README.md\\"\\nrequires-python = \\">=3.11\\"\\nlicense = { text = \\"Apache-2.0\\" }\\nauthors = [{ name = \\"vstaln\\" }]\\ndependencies = [\\n  \\"pyyaml>=6.0\\",\\n  \\"regex>=2024.0\\",\\n  \\"jsonschema>=4.20\\",\\n]\\n\\n[project.optional-dependencies]\\ndev = [\\"pytest>=8.0\\"]\\ntrain = [\\n  \\"torch>=2.0\\",\\n  \\"transformers>=4.40\\",\\n  \\"accelerate>=0.30\\",\\n  \\"bitsandbytes>=0.43\\",\\n  \\"datasets>=2.19\\",\\n  \\"safetensors>=0.4\\",\\n  \\"numpy>=1.26\\",\\n  \\"pyarrow>=15.0\\",\\n  \\"scikit-learn>=1.4\\",\\n]\\n\\n[project.scripts]\\nslopdet = \\"slopdet.cli:main\\"\\nitais = \\"slopdet.cli:main\\"\\n\\n[tool.setuptools.packages.find]\\nwhere = [\\"src\\"]\\n\\n[tool.setuptools.package-data]\\nslopdet = [\\"py.typed\\"]\\n\\n[tool.pytest.ini_options]\\ntestpaths = [\\"tests\\"]\\npythonpath = [\\"src\\"]\\naddopts = \\"-q\\"\\n", "ontology/schema.json": "{\\n  \\"$schema\\": \\"https://json-schema.org/draft/2020-12/schema\\",\\n  \\"$id\\": \\"https://slopdet.local/ontology/schema.json\\",\\n  \\"title\\": \\"Slop pattern\\",\\n  \\"description\\": \\"One frozen ontology entry. Ids are append-only. Disable with enabled=false; never reuse a deleted id.\\",\\n  \\"type\\": \\"object\\",\\n  \\"additionalProperties\\": false,\\n  \\"required\\": [\\n    \\"id\\",\\n    \\"lane\\",\\n    \\"unit\\",\\n    \\"detector\\",\\n    \\"pattern\\",\\n    \\"fix\\",\\n    \\"source\\",\\n    \\"license\\",\\n    \\"min_len_words\\",\\n    \\"paper\\",\\n    \\"enabled\\"\\n  ],\\n  \\"properties\\": {\\n    \\"id\\": {\\n      \\"type\\": \\"string\\",\\n      \\"pattern\\": \\"^[a-z][a-z0-9_]*$\\",\\n      \\"description\\": \\"Stable snake_case id. Never reused.\\"\\n    },\\n    \\"lane\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"style\\", \\"rhetorical\\", \\"construction\\"]\\n    },\\n    \\"unit\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"span\\", \\"sentence\\", \\"paragraph\\", \\"piece\\"]\\n    },\\n    \\"detector\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"regex\\", \\"heuristic\\", \\"model_only\\"]\\n    },\\n    \\"pattern\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 1,\\n      \\"description\\": \\"regex module pattern. Heuristic/model_only entries still ship a compiling proxy.\\"\\n    },\\n    \\"fix\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 8,\\n      \\"description\\": \\"Few-word rewrite instruction. A pattern with no fix is not evidence.\\"\\n    },\\n    \\"source\\": {\\n      \\"type\\": \\"string\\",\\n      \\"minLength\\": 1\\n    },\\n    \\"license\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\"MIT-compatible\\", \\"CC-BY-SA-4.0\\"]\\n    },\\n    \\"min_len_words\\": {\\n      \\"type\\": \\"integer\\",\\n      \\"minimum\\": 0\\n    },\\n    \\"paper\\": {\\n      \\"type\\": [\\"string\\", \\"null\\"]\\n    },\\n    \\"enabled\\": {\\n      \\"type\\": \\"boolean\\"\\n    }\\n  }\\n}\\n", "ontology/patterns.core.yaml": "- id: binary_contrast\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:it(?:[\'\'\\u2019]s| is) not(?: just| only)?.{0,80}?\\\\b(?:it[\'\'\\u2019]s|it is|but)\\\\b|not (?:just|only)\\\\b.{0,60}?\\\\bbut(?: also)?\\\\b|the question isn[\'\'\\u2019]?t\\\\b.{0,60}?\\\\bit[\'\'\\u2019]s\\\\b)\'\\n  fix: State Y directly. Drop the not-X-but-Y frame.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: throat_clearing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:here[\'\\u2019]?s the thing|here[\'\\u2019]?s what I mean|let me be clear|I[\'\\u2019]ll be honest|the uncomfortable truth is|here[\'\\u2019]?s the deal)\\\\b\\n  fix: Cut the opener. Start on the point.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: faux_insight\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:this is the part most people skip|what most people get wrong|here[\'\\u2019]s what nobody tells you|the part everyone misses|what nobody tells you)\\\\b\\n  fix: Drop the lone-expert setup. Make the claim stand alone.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: colon_reveal\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?m)^[A-Z][^.!?\\\\n]{2,60}: [a-z]\'\\n  fix: Rewrite as a plain sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: superficial_analysis\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i),\\\\s+(?:highlighting|underscoring|reflecting|showcasing|emphasizing|ensuring|symbolizing|demonstrating)\\\\b\\n  fix: Replace the trailing -ing clause with a concrete consequence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: importance_puffery\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:stands as a testament|marks a pivotal moment|plays a vital role|solidifies its position|underscores its significance|crucial role|key turning point)\\\\b\\n  fix: State the fact. Let the reader judge if it matters.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: interpretive_metadiscourse\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:that last part matters(?: more than it sounds)?|the key point is|as you can see|this distinction matters|in other words)\\\\b\'\\n  fix: Delete the aside, or replace it with a fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: weasel_attribution\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:experts agree|industry reports suggest|many argue|widely regarded as|studies show|observers have cited|some critics argue)\\\\b\\n  fix: Name the source or cut the claim.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: fake_strong_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as a|functions as a|acts as a|operates as a|stands as a)\\\\b\\n  fix: Use is/has, or name the actual action.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: synonym_cycling\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:the agent|the assistant|the tool|the system|the platform)\\\\b.{0,180}\\\\b(?:the agent|the assistant|the tool|the system|the platform)\\\\b\\n  fix: Repeat the clear word. Do not rotate synonyms for style.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: negative_listing\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bnot a\\\\b.{0,50}\\\\bnot a\\\\b.{0,50}\\\\b(?:a |an |the )\\n  fix: Just say Z. Drop the not-X not-Y list.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: dramatic_fragmentation\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)(?:\\\\bthat[\'\\u2019]s it\\\\. that[\'\\u2019]s the whole thing\\\\b|(?m)^And [A-Z][^.!?\\\\n]{0,50}\\\\.\\\\s*\\\\nAnd )\\n  fix: Use complete sentences. Stop stacking And-fragments.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: robotic_rhythm\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?m)(?:^[A-Z][^.!?\\\\n]{10,42}\\\\.\\\\s+){2}[A-Z][^.!?\\\\n]{10,42}\\\\.\\n  fix: Vary sentence length and shape. Merge or split one of the three.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 40\\n  paper: null\\n  enabled: true\\n- id: rhetorical_setup\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:what if I told you|think about it:|plot twist:)\\\\b\\n  fix: Drop the setup. Make the point.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: fake_profound_kicker\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:and that(?:[\'\\u2019]s| is) the (?:whole |real )?point|the rest is (?:just )?noise|that(?:[\'\\u2019]s| is) the whole game)\\\\b\\n  fix: Delete the mic-drop. End on the last concrete sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: recap_ending\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?im)(?:^|(?<=[.!?]\\\\s))(?:in conclusion|ultimately|overall|to sum up|in summary|to conclude)\\\\s*[,:]\\n  fix: End on the last concrete point. Do not restate the piece.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: formatting_slop\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)^#{1,3}\\\\s+.+\\\\n(?:.*\\\\n){0,2}^#{1,3}\\\\s+\\n  fix: Drop headers over two-sentence sections. Write prose.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: em_dash_cluster\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \\u2014[^.\\\\n]{0,90}\\u2014\\n  fix: Use a comma, period, or parentheses. Do not cluster dashes.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: ban_delve_class\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:delve|delves|delving|foster|fostering|leverage|leveraging|utilize|utilizing|facilitate|facilitating|empower|empowering|streamline|streamlining)\\\\b\\n  fix: Name the action. Use a concrete verb.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_puffery_noun\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:tapestry|realm|beacon|paradigm(?: shift)?)\\\\b\'\\n  fix: Replace the metaphor with the actual thing.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_puffery_adj\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:robust|cutting-edge|multifaceted|meticulous(?:ly)?|intricate|intricacies|paramount|transformative|vibrant|pivotal|groundbreaking|seamless(?:ly)?)\\\\b\\n  fix: Cut the adjective, or name the property it stands in for.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_journey_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:elevate|elevating|embark|embarking|supercharge|supercharging|harness|harnessing|ever-evolving)\\\\b\\n  fix: Use a plain verb. Say what actually happens.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_showcase_verb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:underscore|underscores|underscoring|showcase|showcases|showcasing|highlight|highlights|highlighting|emphasize|emphasizes|emphasizing)\\\\b\\n  fix: State the fact. Do not announce its importance.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: ban_corporate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:synergy|synergies|pain points?|value proposition|thought leaders?(?:hip)?|circle back|touch base|move the needle)\\\\b\\n  fix: Say the work in ordinary words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: ban_game_changer\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:game[- ]chang(?:er|ing)|this is huge|this changes everything|unlock(?:s|ing)? the power)\\\\b\\n  fix: Name the change. Drop the slogan.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: empty_adverb\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:literally|honestly|simply|actually|truly|fundamentally|importantly|crucially|inherently|inevitably)\\\\b\\n  fix: Cut the adverb if it adds nothing.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_worth_noting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit[\'\\u2019]?s worth noting\\\\b\\n  fix: Delete. Start with the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_important_to_note\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\bit[\'\'\\u2019]?s important to note(?: that)?\\\\b\'\\n  fix: Delete. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_end_of_the_day\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bat the end of the day\\\\b\\n  fix: Cut the proverb. Make the claim.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_when_it_comes_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwhen it comes to\\\\b\\n  fix: Name the subject and start.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_at_its_core\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bat its core\\\\b\\n  fix: Drop the frame. State the mechanism.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_todays\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin today[\'\\u2019]?s\\\\b\\n  fix: Cut the era opener. Name the situation.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_the_age_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the age of\\\\b\\n  fix: Cut the era opener.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_the_world_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the world of\\\\b\\n  fix: Name the field. Skip the tour.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_the_reality_is\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthe reality is\\\\b\\n  fix: Drop the drumroll. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_the_truth_is\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthe truth is\\\\b\\n  fix: Drop the drumroll. State the fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_terms_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin terms of\\\\b\\n  fix: Rewrite with a direct object.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_with_regard_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwith regard to\\\\b\\n  fix: Name the topic and continue.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_order_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin order to\\\\b\\n  fix: Rewrite as \'to\' plus the verb.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_going_forward\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bgoing forward\\\\b\\n  fix: Cut it, or name the date.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_this_article\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin this article\\\\b\\n  fix: Do not announce the article.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_lets_dive_in\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\blet[\'\\u2019]?s dive (?:in|deeper|into)\\\\b\\n  fix: Start the first fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_unlock_the_power\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bunlock(?:s|ing)? the power of\\\\b\\n  fix: Name the action the reader can take.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_bridge_the_gap\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bbridge(?:s|ing)? the gap\\\\b\\n  fix: Name the two sides and the actual link.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_i_hope_this_helps\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bI hope this helps\\\\b\\n  fix: End on the last useful sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_i_hope_this_finds_you\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'(?i)\\\\bI hope this(?: email)? finds you well\\\\b\'\\n  fix: Open with the reason you wrote.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_whether_youre\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwhether you(?:[\'\\u2019]re| are) a\\\\b.{0,40}\\\\bor a\\\\b\\n  fix: Pick one reader. Write to them.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_from_x_to_y_opener\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^From .{2,40} to .{2,40}[,.]\\n  fix: Open with the specific case, not a range.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_this_is_where_x_comes_in\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bthis is where\\\\b.{0,40}\\\\bcomes in\\\\b\\n  fix: Introduce the thing without the drumroll.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_firstly_secondly_thirdly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:firstly|secondly|thirdly)\\\\b\\n  fix: Use 1. 2. 3. or just write the points.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_without_further_ado\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bwithout further ado\\\\b\\n  fix: Cut the drumroll and start.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_in_a_nutshell\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin a nutshell\\\\b\\n  fix: State the summary as a sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_please_dont_hesitate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bplease don[\'\\u2019]?t hesitate to (?:reach out|contact)\\\\b\\n  fix: Give the actual next step.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: phrase_not_just_x_but_y\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit[\'\\u2019]?s not just about\\\\b.{0,50}\\\\bit[\'\\u2019]?s about\\\\b\\n  fix: State Y. Drop the not-just frame.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: rule_of_three\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b\\\\w+,\\\\s+\\\\w+,\\\\s+and\\\\s+\\\\w+\\\\b\\n  fix: Break the default trio. Use two, four, or one.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 20\\n  paper: null\\n  enabled: true\\n- id: uniform_sentence_length\\n  lane: construction\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?s)(?=.{80,})\\n  fix: Mix a short sentence with a long one.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: null\\n  enabled: true\\n- id: parataxis\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?m)(?:^[A-Z][^.!?\\\\n]{0,28}\\\\.\\\\s+){2}[A-Z][^.!?\\\\n]{0,28}\\\\.\\n  fix: Connect the thoughts. Add a because, but, or which.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: hedging_seesaw\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\bon the one hand\\\\b.{0,200}\\\\bon the other hand\\\\b\\n  fix: Pick a side. Give the counterpoint one sentence.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: corporate_pep_talk\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:together we can|exciting opportunity|passionate about|unlock(?:ing)? potential|drive(?:s|ing)? (?:impact|outcomes)|deliver(?:ing)? value)\\\\b\\n  fix: Write like someone who did the work, including the mess.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: identical_paragraph_structure\\n  lane: construction\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?s)(?=.{200,})\\n  fix: Break the topic-explain-example-transition mold.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 200\\n  paper: null\\n  enabled: true\\n- id: bullet_overuse\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)(?:^[\\\\t ]*(?:[-*]|\\\\d+\\\\.)\\\\s+.+\\\\n){6,}\\n  fix: Turn the list into sentences, or cap it at five.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: as_role_opener\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^As an? [A-Z][^,.]{2,40},\\\\s+I\\\\b\\n  fix: Say the thing. Do not announce credentials.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: cross_section_parallelism\\n  lane: construction\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?s)(?=.{300,})\\n  fix: Give each section a different shape and length.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 300\\n  paper: null\\n  enabled: true\\n- id: passive_construction\\n  lane: style\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:is being \\\\w+ed|was found to be|are considered to be|has been shown to)\\\\b\\n  fix: Write the actor and the verb.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: mandatory_paragraph_transition\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)(?:^|\\\\n)\\\\s*(?:moreover|furthermore|additionally|in addition|that said|with that in mind)\\\\s*,\\n  fix: Let some paragraphs just stop.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: punct_em_dash_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:\\u2014.*){2,}\\n  fix: At most one em dash per 500 words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 1\\n  paper: null\\n  enabled: true\\n- id: punct_exclamation_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:!.*){2,}\\n  fix: At most one exclamation per 1,000 words.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 1\\n  paper: null\\n  enabled: true\\n- id: punct_ellipsis_budget\\n  lane: style\\n  unit: piece\\n  detector: heuristic\\n  pattern: (?:\\\\.{3}|\\u2026).*(?:\\\\.{3}|\\u2026)\\n  fix: One ellipsis per piece, only for a real trailing-off.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_certainly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^(?:certainly|absolutely|sure|great question|that[\'\\u2019]s a great point)[,!]\\n  fix: Answer. Skip the cheer.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_moreover\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Moreover,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_furthermore\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Furthermore,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_additionally\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Additionally,\\n  fix: Start with the next fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: opener_interestingly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Interestingly,\\n  fix: State the interesting fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_notably\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Notably,\\n  fix: State the fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_importantly\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Importantly,\\n  fix: State the fact. Drop the label.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_indeed\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?im)^Indeed,\\n  fix: Continue without the nod.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: opener_as_an_ai\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:as an AI|as a language model)\\\\b\\n  fix: Never announce the model.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: emoji_bullet\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^[\\\\t ]*[\\u2705\\ud83d\\udd25\\u2728\\ud83d\\udca1\\ud83d\\udc49\\u2b50\\ufe0f\\u2b50]\\\\s\\n  fix: Write a sentence. Do not emoji-bullet.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: hashtag_stack\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?:#[A-Za-z0-9_]+)(?:\\\\s+#[A-Za-z0-9_]+){2,}\\n  fix: Zero to two hashtags, in the sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: markdown_in_plain\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^\\\\*\\\\*[^*]{3,40}\\\\*\\\\*\\\\s*$\\n  fix: Do not bold a whole line for emphasis.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: copula_avoidance_surface\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as|stands as|functions as|operates as|boasts a|features a)\\\\b\\n  fix: Use is or has when that is what you mean.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: in_the_realm_of\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin the realm of\\\\b\\n  fix: Name the field.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: a_testament_to\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\ba testament to\\\\b\\n  fix: State what happened.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: rest_assured\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\brest assured\\\\b\\n  fix: Give the actual guarantee or drop it.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: it_goes_without_saying\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bit goes without saying\\\\b\\n  fix: If it goes without saying, delete it.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: in_essence\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin essence\\\\b\\n  fix: State the claim.\\n  source: anti-ai-slop-writing\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: please_note_that\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bplease note that\\\\b\\n  fix: State the note as a fact.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: as_mentioned_earlier\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bas (?:mentioned|noted|discussed) earlier\\\\b\\n  fix: Repeat the fact if needed. Do not point at the earlier sentence.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: in_todays_digital_age\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bin today[\'\\u2019]?s (?:fast-paced |ever-changing |digital )?world\\\\b\\n  fix: Name the actual constraint.\\n  source: no-ai-slop\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n", "ontology/patterns.wikipedia.yaml": "# SPDX-License-Identifier: CC-BY-SA-4.0\\n#\\n# Derived from Wikipedia:Signs of AI writing\\n# https://en.wikipedia.org/wiki/Wikipedia:Signs_of_AI_writing\\n# License: CC BY-SA 4.0\\n# https://creativecommons.org/licenses/by-sa/4.0/\\n#\\n# Share-alike applies to the descriptive text (fix blurbs) in this file.\\n# Regex strings are functional. Do not paste these descriptions into Apache-2.0 source.\\n#\\n- id: wiki_significance_puffery\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:stands as a testament|marking a pivotal moment|reflects broader|symbolizing its (?:ongoing|enduring|lasting)|setting the stage for|indelible mark|evolving landscape|focal point|deeply rooted)\\\\b\\n  fix: Cut the legacy sermon. Keep the dated fact.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_notability_boilerplate\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:independent coverage|active social media presence|profiled in|widely-read outlets|significant, substantial, secondary coverage)\\\\b\\n  fix: Cite the source. Do not recite notability policy.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_promotional_tone\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:nestled (?:in|within)|in the heart of|rich (?:cultural )?heritage|natural beauty|diverse array|boasts a|renowned for)\\\\b\\n  fix: Drop the brochure. Name one specific place or fact.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_ai_vocabulary_cluster\\n  lane: style\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)(?:\\\\b(?:delve|tapestry|underscore|pivotal|vibrant|intricate|meticulous|landscape|testament|showcase|foster|align with)\\\\b.*){3,}\\n  fix: One inflated word can be accident. Three in one passage is the tell. Rewrite with plain nouns.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: kobak-2406.07016\\n  enabled: true\\n- id: wiki_copula_avoidance\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:serves as|stands as|marks a|functions as|operates as|holds the distinction of being|refers to)\\\\b\\n  fix: Use is or are. Stop dressing the copula.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: wiki_negative_parallelism\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: \'(?i)\\\\b(?:not only .{0,40} but(?: also)?|it is not .{0,40}, it(?:[\'\'\\u2019]s| is)|rather than .{0,30}$)\'\\n  fix: Drop the misconception-clearing frame. State the property.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_challenges_future\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?i)\\\\b(?:challenges remain|future prospects|looking ahead|as .+ continues to evolve|more research is needed)\\\\b\\n  fix: Stop the outline close. End on what is known now.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_awards_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)^#+\\\\s+Awards and recognition\\\\s*$\\n  fix: Merge awards into the career section if they are few.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_title_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#\\\\s+[A-Z].{0,80}\\\\n\\\\n\\n  fix: Do not repeat the article title as the first heading.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_title_case_heading\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#{2,3}\\\\s+(?:[A-Z][a-z]+\\\\s+){2,}[A-Z][a-z]+$\\n  fix: Use sentence case in headings.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_boldface_overuse\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?:\\\\*\\\\*[^*]{2,40}\\\\*\\\\*.*){3,}\\n  fix: Bold once, if at all. Stop sprinkling emphasis.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_inline_header_list\\n  lane: style\\n  unit: paragraph\\n  detector: regex\\n  pattern: (?m)^[-*]\\\\s+\\\\*\\\\*[^*]{2,40}\\\\*\\\\*:\\\\s\\n  fix: Turn canned bold-label bullets into sentences.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_emoji_formatting\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^[\\\\t ]*(?:[\\u2705\\u274c\\u26a0\\ufe0f\\ud83d\\udccc\\ud83d\\udd0d\\ud83d\\udca1]|:[a-z_]+:)\\\\s\\n  fix: No emoji as structure.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_curly_quotes\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: \'[\\u201c\\u201d\\u2018\\u2019].{0,80}[\\u201c\\u201d\\u2018\\u2019]\'\\n  fix: Straight quotes unless the house style needs curls.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_collaborative_you\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:I hope this helps|let me know if|I[\'\\u2019]d be happy to|as you requested)\\\\b\\n  fix: This is article text, not a chat reply. Cut the assistant voice.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_knowledge_cutoff\\n  lane: style\\n  unit: sentence\\n  detector: regex\\n  pattern: (?i)\\\\b(?:as of my last (?:training|update)|I don[\'\\u2019]t have (?:access|information)|my knowledge cutoff)\\\\b\\n  fix: Delete the model disclaimer. Write from sources.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_placeholder_text\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\b(?:TODO|TBD|lorem ipsum|insert (?:text|citation|source) here|\\\\[placeholder\\\\])\\\\b\\n  fix: Replace placeholders before the text ships.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_skipping_heading_levels\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?m)^#\\\\s+.+\\\\n+(?:#{3,}\\\\s+)\\n  fix: Do not skip from H1 to H3.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_thematic_break_spam\\n  lane: style\\n  unit: piece\\n  detector: regex\\n  pattern: (?m)(?:^---\\\\s*\\\\n){2,}\\n  fix: Horizontal rules are not sectioning.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n- id: wiki_valuable_insights\\n  lane: style\\n  unit: span\\n  detector: regex\\n  pattern: (?i)\\\\bvaluable insights\\\\b\\n  fix: Name the finding. Insights is empty.\\n  source: wikipedia-signs-of-ai-writing\\n  license: CC-BY-SA-4.0\\n  min_len_words: 0\\n  paper: null\\n  enabled: true\\n", "ontology/patterns.rhetorical.yaml": "- id: rhet_present_participial\\n  lane: rhetorical\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i),\\\\s+\\\\w+ing\\\\b.{0,80}(?:\\\\.|$)\\n  fix: The comma-VBG tag is a proxy for present-participial clauses. Replace with a finite clause.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_nominalization_density\\n  lane: rhetorical\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\b\\\\w{4,}(?:tion|sion|ment|ness|ity)s?\\\\b\\n  fix: High -tion/-ment/-ness/-ity rate. Prefer verbs over abstract nouns.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_copula_avoidance\\n  lane: rhetorical\\n  unit: paragraph\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:is|are|was|were|be|been|being)\\\\b\\n  fix: Low be-verb ratio vs lexical verbs is the tell. Restore is/are where they are clearer.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 80\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_that_complement\\n  lane: rhetorical\\n  unit: sentence\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:said|argued|claimed|noted|reported|suggested|found|showed|believed) that\\\\b\\n  fix: That-complement rate vs human baseline. Keep that when it prevents a garden path.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n- id: rhet_adj_stacking\\n  lane: rhetorical\\n  unit: span\\n  detector: heuristic\\n  pattern: (?i)\\\\b(?:a|an|the)\\\\s+[A-Za-z]+,\\\\s+[A-Za-z]+(?:,|\\\\s+and)\\\\s+[A-Za-z]+\\\\b\\n  fix: Stacked attributive adjectives. Keep one modifier that earns its place.\\n  source: reinhart-biber\\n  license: MIT-compatible\\n  min_len_words: 0\\n  paper: reinhart-2410.16107\\n  enabled: true\\n", "src/slopdet/__init__.py": "\\"\\"\\"ITAIS (Is This AI Slop?) \\u2014 checkable-hit + resemblance detector. Import: slopdet.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "src/slopdet/ontology.py": "\\"\\"\\"Load and freeze the slop pattern ontology.\\n\\nIds are append-only. Disable a pattern with enabled=false; do not delete it.\\nONTOLOGY_SHA256 is sha256 of the three YAML files concatenated in this order:\\npatterns.core.yaml, patterns.wikipedia.yaml, patterns.rhetorical.yaml.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport regex as regex_mod\\nimport yaml\\nfrom jsonschema import Draft202012Validator\\n\\nYAML_NAMES = (\\n    \\"patterns.core.yaml\\",\\n    \\"patterns.wikipedia.yaml\\",\\n    \\"patterns.rhetorical.yaml\\",\\n)\\n\\n\\ndef default_ontology_dir() -> Path:\\n    return Path(__file__).resolve().parents[2] / \\"ontology\\"\\n\\n\\nclass OntologyError(ValueError):\\n    \\"\\"\\"Invalid ontology data.\\"\\"\\"\\n\\n\\n@dataclass(frozen=True)\\nclass Pattern:\\n    id: str\\n    lane: str\\n    unit: str\\n    detector: str\\n    pattern: str\\n    fix: str\\n    source: str\\n    license: str\\n    min_len_words: int\\n    paper: str | None\\n    enabled: bool\\n    compiled: Any = field(repr=False, compare=False)\\n\\n\\n@dataclass(frozen=True)\\nclass Ontology:\\n    patterns: tuple[Pattern, ...]\\n    sha256: str\\n    by_id: dict[str, Pattern]\\n\\n    @property\\n    def ONTOLOGY_SHA256(self) -> str:\\n        return self.sha256\\n\\n    def enabled_patterns(self) -> tuple[Pattern, ...]:\\n        return tuple(p for p in self.patterns if p.enabled)\\n\\n\\ndef _schema(ontology_dir: Path) -> dict[str, Any]:\\n    return json.loads((ontology_dir / \\"schema.json\\").read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _load_yaml_list(path: Path) -> list[dict[str, Any]]:\\n    data = yaml.safe_load(path.read_text(encoding=\\"utf-8\\"))\\n    if not isinstance(data, list):\\n        raise OntologyError(f\\"{path.name} must be a YAML list, got {type(data).__name__}\\")\\n    return data\\n\\n\\ndef load_ontology(ontology_dir: Path | None = None) -> Ontology:\\n    ontology_dir = Path(ontology_dir) if ontology_dir else default_ontology_dir()\\n    validator = Draft202012Validator(_schema(ontology_dir))\\n    seen: dict[str, str] = {}\\n    patterns: list[Pattern] = []\\n    raw_parts: list[bytes] = []\\n\\n    for name in YAML_NAMES:\\n        path = ontology_dir / name\\n        raw_parts.append(path.read_bytes())\\n        for i, entry in enumerate(_load_yaml_list(path)):\\n            errors = sorted(validator.iter_errors(entry), key=lambda e: list(e.path))\\n            if errors:\\n                raise OntologyError(f\\"{name}[{i}]: {errors[0].message}\\")\\n            pid = entry[\\"id\\"]\\n            if pid in seen:\\n                raise OntologyError(f\\"duplicate id {pid!r} in {name} and {seen[pid]}\\")\\n            seen[pid] = name\\n            try:\\n                compiled = regex_mod.compile(entry[\\"pattern\\"])\\n            except regex_mod.error as exc:\\n                raise OntologyError(f\\"{pid}: regex does not compile: {exc}\\") from exc\\n            patterns.append(\\n                Pattern(\\n                    id=pid,\\n                    lane=entry[\\"lane\\"],\\n                    unit=entry[\\"unit\\"],\\n                    detector=entry[\\"detector\\"],\\n                    pattern=entry[\\"pattern\\"],\\n                    fix=entry[\\"fix\\"],\\n                    source=entry[\\"source\\"],\\n                    license=entry[\\"license\\"],\\n                    min_len_words=int(entry[\\"min_len_words\\"]),\\n                    paper=entry[\\"paper\\"],\\n                    enabled=bool(entry[\\"enabled\\"]),\\n                    compiled=compiled,\\n                )\\n            )\\n\\n    sha256 = hashlib.sha256(b\\"\\".join(raw_parts)).hexdigest()\\n    frozen = tuple(patterns)\\n    return Ontology(\\n        patterns=frozen,\\n        sha256=sha256,\\n        by_id={p.id: p for p in frozen},\\n    )\\n\\n\\nONTOLOGY_SHA256: str | None = None\\n\\n\\ndef ontology_sha256(ontology_dir: Path | None = None) -> str:\\n    global ONTOLOGY_SHA256\\n    if ONTOLOGY_SHA256 is None:\\n        ONTOLOGY_SHA256 = load_ontology(ontology_dir).sha256\\n    return ONTOLOGY_SHA256\\n", "src/slopdet/weaklabel.py": "\\"\\"\\"Span weak-labeler. Overlapping hits from different ids all survive.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nfrom slopdet.ontology import Ontology, Pattern\\n\\n\\ndef _word_count(text: str) -> int:\\n    return len(text.split())\\n\\n\\ndef label_text(text: str, ontology: Ontology, *, enabled_only: bool = True) -> list[dict[str, Any]]:\\n    n_words = _word_count(text)\\n    patterns: tuple[Pattern, ...] = (\\n        ontology.enabled_patterns() if enabled_only else ontology.patterns\\n    )\\n    hits: list[dict[str, Any]] = []\\n    for pattern in patterns:\\n        if n_words < pattern.min_len_words:\\n            continue\\n        for match in pattern.compiled.finditer(text):\\n            start, end = match.start(), match.end()\\n            if end <= start:\\n                continue\\n            hits.append(\\n                {\\n                    \\"id\\": pattern.id,\\n                    \\"start\\": start,\\n                    \\"end\\": end,\\n                    \\"unit\\": pattern.unit,\\n                    \\"lane\\": pattern.lane,\\n                    \\"quote\\": text[start:end],\\n                    \\"fix\\": pattern.fix,\\n                }\\n            )\\n    hits.sort(key=lambda h: (h[\\"start\\"], h[\\"end\\"], h[\\"id\\"]))\\n    return hits\\n", "src/slopdet/construction.py": "\\"\\"\\"Cheap deterministic construction stats. No spaCy, no GPU.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nfrom typing import Any\\n\\n_SENT_SPLIT = re.compile(r\\"(?<=[.!?])\\\\s+\\")\\n_PARA_SPLIT = re.compile(r\\"\\\\n\\\\s*\\\\n\\")\\n_WORD = re.compile(r\\"[A-Za-z\']+\\")\\n_PROPER = re.compile(r\\"\\\\b[A-Z][a-z]{2,}\\\\b\\")\\n_DIGIT = re.compile(r\\"\\\\d\\")\\n_DATE = re.compile(\\n    r\\"\\\\b(?:(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\\\\.?\\\\s+\\\\d{1,2}\\"\\n    r\\"|\\\\d{1,2}/\\\\d{1,2}/\\\\d{2,4}|\\\\d{4})\\\\b\\",\\n    re.I,\\n)\\n_SUBORD = re.compile(\\n    r\\"\\\\b(?:because|although|though|unless|while|whereas|if|when|after|before|since)\\\\b\\",\\n    re.I,\\n)\\n_CLOSURE = re.compile(\\n    r\\"\\\\b(?:in conclusion|ultimately|overall|to sum up|in summary|to conclude)\\\\b\\",\\n    re.I,\\n)\\n_OVER_EXPLAIN = re.compile(\\n    r\\"\\\\b(?:the key point is|as you can see|this distinction matters|in other words|\\"\\n    r\\"highlighting|underscoring|reflecting|showcasing)\\\\b\\",\\n    re.I,\\n)\\n\\n\\ndef _sentences(text: str) -> list[str]:\\n    parts = [s.strip() for s in _SENT_SPLIT.split(text.strip()) if s.strip()]\\n    return parts or ([text.strip()] if text.strip() else [])\\n\\n\\ndef _paragraphs(text: str) -> list[str]:\\n    parts = [p.strip() for p in _PARA_SPLIT.split(text) if p.strip()]\\n    return parts or ([text.strip()] if text.strip() else [])\\n\\n\\ndef _cosine(a: list[float], b: list[float]) -> float:\\n    dot = sum(x * y for x, y in zip(a, b))\\n    na = math.sqrt(sum(x * x for x in a))\\n    nb = math.sqrt(sum(y * y for y in b))\\n    if na == 0 or nb == 0:\\n        return 0.0\\n    return dot / (na * nb)\\n\\n\\ndef _para_features(para: str) -> list[float]:\\n    sents = _sentences(para)\\n    lengths = [len(s.split()) for s in sents] or [0]\\n    words = _WORD.findall(para.lower())\\n    n = max(len(words), 1)\\n    mean_len = sum(lengths) / max(len(lengths), 1)\\n    ttr = len(set(words)) / n\\n    comma = para.count(\\",\\") / n\\n    sub = len(_SUBORD.findall(para)) / n\\n    mean_word = sum(len(w) for w in words) / n\\n    return [mean_len, ttr, comma, sub, mean_word]\\n\\n\\ndef construction_stats(text: str) -> dict[str, Any]:\\n    sents = _sentences(text)\\n    lengths = [len(s.split()) for s in sents]\\n    mean = (sum(lengths) / len(lengths)) if lengths else 0.0\\n    if len(lengths) >= 2:\\n        var = sum((x - mean) ** 2 for x in lengths) / len(lengths)\\n        std = math.sqrt(var)\\n        burstiness = std / mean if mean else 0.0\\n        adjacent_contrast = sum(\\n            1 for a, b in zip(lengths, lengths[1:]) if abs(a - b) >= 20\\n        )\\n    else:\\n        burstiness = 0.0\\n        adjacent_contrast = 0\\n\\n    paras = _paragraphs(text)\\n    vecs = [_para_features(p) for p in paras]\\n    if len(vecs) >= 2:\\n        sims = [\\n            _cosine(vecs[i], vecs[j])\\n            for i in range(len(vecs))\\n            for j in range(i + 1, len(vecs))\\n        ]\\n        evenness = sum(sims) / len(sims)\\n    else:\\n        evenness = 0.0\\n\\n    if paras:\\n        last = paras[-1]\\n        earlier_openers = \\" \\".join(_sentences(p)[:1][0] if _sentences(p) else \\"\\" for p in paras[:-1])\\n        last_words = set(_WORD.findall(last.lower()))\\n        earlier_words = set(_WORD.findall(earlier_openers.lower()))\\n        overlap = len(last_words & earlier_words) / max(len(last_words), 1)\\n        recap_closure = overlap + (0.5 if _CLOSURE.search(last) else 0.0)\\n    else:\\n        recap_closure = 0.0\\n\\n    n_words = max(len(text.split()), 1)\\n    over_explain = 1000.0 * len(_OVER_EXPLAIN.findall(text)) / n_words\\n\\n    portable = 0\\n    for sent in sents:\\n        if not (_PROPER.search(sent) or _DIGIT.search(sent) or _DATE.search(sent)):\\n            portable += 1\\n    portability = portable / max(len(sents), 1)\\n\\n    return {\\n        \\"burstiness\\": round(burstiness, 6),\\n        \\"adjacent_contrast\\": int(adjacent_contrast),\\n        \\"evenness\\": round(evenness, 6),\\n        \\"recap_closure\\": round(recap_closure, 6),\\n        \\"over_explain\\": round(over_explain, 6),\\n        \\"portability\\": round(portability, 6),\\n        \\"n_sentences\\": len(sents),\\n        \\"n_paragraphs\\": len(paras),\\n        \\"n_words\\": n_words,\\n    }\\n", "src/slopdet/report.py": "\\"\\"\\"Hit renderer. Two lanes never merge. Never emit a percentage-of-AI string.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nFORBIDDEN_SUBSTRINGS = (\\n    \\"% AI\\",\\n    \\"% ai\\",\\n    \\"percent AI\\",\\n    \\"AI-generated\\",\\n    \\"ai-generated\\",\\n    \\"written by ChatGPT\\",\\n    \\"written by GPT\\",\\n)\\n\\n\\ndef render_hits(\\n    hits: list[dict[str, Any]],\\n    *,\\n    resemblance: dict[str, Any] | None = None,\\n) -> dict[str, Any]:\\n    style = [h for h in hits if h.get(\\"lane\\") != \\"resemblance\\"]\\n    out: dict[str, Any] = {\\n        \\"status\\": \\"ok\\",\\n        \\"hits\\": [\\n            {\\n                \\"id\\": h[\\"id\\"],\\n                \\"lane\\": h.get(\\"lane\\", \\"style\\"),\\n                \\"unit\\": h.get(\\"unit\\", \\"span\\"),\\n                \\"quote\\": h.get(\\"quote\\", \\"\\"),\\n                \\"fix\\": h.get(\\"fix\\", \\"\\"),\\n            }\\n            for h in style\\n        ],\\n        \\"style_summary\\": \\"Nothing matched.\\" if not style else None,\\n        \\"resemblance\\": None,\\n    }\\n    if not style:\\n        out[\\"style_summary\\"] = \\"Nothing matched.\\"\\n    if resemblance is not None:\\n        pct = resemblance.get(\\"human_percentile\\")\\n        if pct is None:\\n            out[\\"resemblance\\"] = {\\n                \\"label\\": \\"matches_ai_pile\\",\\n                \\"text\\": resemblance.get(\\"text\\", \\"Resemblance unavailable.\\"),\\n            }\\n        else:\\n            out[\\"resemblance\\"] = {\\n                \\"label\\": \\"matches_ai_pile\\",\\n                \\"text\\": (\\n                    f\\"Resembles the AI pile more than {pct:.0f}% of human reference texts.\\"\\n                ),\\n            }\\n    blob = str(out)\\n    for bad in FORBIDDEN_SUBSTRINGS:\\n        if bad in blob and \\"AI pile\\" not in bad:\\n            raise ValueError(f\\"forbidden copy leaked: {bad!r}\\")\\n    return out\\n", "src/slopdet/verify.py": "\\"\\"\\"Fail-closed artifact verification. Hash mismatch \\u2192 empty hits, no regex fallback.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n\\nclass UnverifiedArtifact(Exception):\\n    def __init__(self, payload: dict[str, Any]):\\n        super().__init__(payload[\\"status\\"])\\n        self.payload = payload\\n\\n\\ndef sha256_file(path: Path) -> str:\\n    h = hashlib.sha256()\\n    with path.open(\\"rb\\") as fh:\\n        for chunk in iter(lambda: fh.read(1024 * 1024), b\\"\\"):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef load_manifest(artifacts_dir: Path) -> dict[str, Any]:\\n    path = artifacts_dir / \\"MANIFEST.json\\"\\n    if not path.is_file():\\n        raise UnverifiedArtifact(unverified_payload(\\"missing_manifest\\"))\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef unverified_payload(reason: str = \\"unverified_artifact\\") -> dict[str, Any]:\\n    return {\\"status\\": \\"unverified_artifact\\", \\"reason\\": reason, \\"hits\\": [], \\"resemblance\\": None}\\n\\n\\ndef verify_artifacts(artifacts_dir: Path) -> dict[str, Any]:\\n    artifacts_dir = Path(artifacts_dir)\\n    try:\\n        manifest = load_manifest(artifacts_dir)\\n    except UnverifiedArtifact:\\n        return unverified_payload(\\"missing_manifest\\")\\n    files = manifest.get(\\"files\\") or {}\\n    for rel, expected in files.items():\\n        path = artifacts_dir / rel\\n        if not path.is_file():\\n            return unverified_payload(f\\"missing:{rel}\\")\\n        actual = sha256_file(path)\\n        if actual != expected:\\n            return unverified_payload(f\\"mismatch:{rel}\\")\\n    return {\\"status\\": \\"ok\\", \\"manifest\\": manifest}\\n", "src/slopdet/calibrate.py": "\\"\\"\\"Calibration helpers for matches_ai_pile (1% FPR on human reference).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\ndef threshold_at_fpr(human_scores: list[float], fpr: float = 0.01) -> float:\\n    if not human_scores:\\n        return 1.0\\n    ordered = sorted(human_scores, reverse=True)\\n    k = max(0, min(len(ordered) - 1, int(len(ordered) * fpr)))\\n    return float(ordered[k])\\n\\n\\ndef human_percentile(score: float, human_scores: list[float]) -> float:\\n    if not human_scores:\\n        return 0.0\\n    n = sum(1 for s in human_scores if score > s)\\n    return 100.0 * n / len(human_scores)\\n\\n\\ndef calibration_record(human_scores: list[float], fpr: float = 0.01) -> dict[str, Any]:\\n    return {\\n        \\"fpr\\": fpr,\\n        \\"threshold\\": threshold_at_fpr(human_scores, fpr),\\n        \\"n_human\\": len(human_scores),\\n        \\"label\\": \\"matches_ai_pile\\",\\n    }\\n", "src/slopdet/cli.py": "\\"\\"\\"CLI: print checkable hits. Never a percentage-of-AI verdict.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nfrom slopdet.construction import construction_stats\\nfrom slopdet.ontology import default_ontology_dir, load_ontology\\nfrom slopdet.report import render_hits\\nfrom slopdet.weaklabel import label_text\\n\\n\\ndef detect(text: str, ontology_dir: Path | None = None) -> dict:\\n    onto = load_ontology(ontology_dir or default_ontology_dir())\\n    hits = label_text(text, onto)\\n    result = render_hits(hits)\\n    result[\\"construction\\"] = construction_stats(text)\\n    result[\\"ontology_sha256\\"] = onto.sha256\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(prog=\\"itais\\")\\n    parser.add_argument(\\"text\\", nargs=\\"?\\", help=\\"Text to scan. Defaults to stdin.\\")\\n    parser.add_argument(\\"--ontology\\", type=Path, default=None)\\n    parser.add_argument(\\"--json\\", action=\\"store_true\\")\\n    args = parser.parse_args(argv)\\n    text = args.text if args.text is not None else sys.stdin.read()\\n    result = detect(text, args.ontology)\\n    if args.json:\\n        json.dump(result, sys.stdout, ensure_ascii=False, indent=2)\\n        sys.stdout.write(\\"\\\\n\\")\\n        return 0\\n    print(result.get(\\"style_summary\\") or \\"\\")\\n    for hit in result[\\"hits\\"]:\\n        print(f\\"- {hit[\'id\']}: {hit[\'quote\']!r}\\")\\n        print(f\\"  fix: {hit[\'fix\']}\\")\\n    if result.get(\\"resemblance\\"):\\n        print(result[\\"resemblance\\"][\\"text\\"])\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "src/slopdet/teacher.py": "\\"\\"\\"Teacher residual cache \\u2014 Colab/T4. Imported only when transformers is installed.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nTEACHER_CANDIDATES = (\\n    \\"google/gemma-3-4b-it\\",\\n    \\"unsloth/gemma-3-4b-it-bnb-4bit\\",\\n    \\"Qwen/Qwen2.5-0.5B-Instruct\\",\\n)\\n", "src/slopdet/student.py": "\\"\\"\\"Tiny Fast-style student. Token-level hidden states for distillation.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\ntry:\\n    import torch\\n    from torch import nn\\nexcept ImportError as exc:  # pragma: no cover\\n    raise ImportError(\\"student.py needs torch. Install the train extra.\\") from exc\\n\\n\\n@dataclass\\nclass StudentConfig:\\n    vocab_size: int\\n    pad_token_id: int\\n    max_length: int = 256\\n    token_embed_dim: int = 128\\n    d_model: int = 256\\n    n_layers: int = 4\\n    n_heads: int = 4\\n    mlp_dim: int = 1024\\n    dropout: float = 0.1\\n    output_dim: int = 2560\\n\\n\\nclass Student(nn.Module):\\n    def __init__(self, cfg: StudentConfig):\\n        super().__init__()\\n        self.cfg = cfg\\n        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.token_embed_dim, padding_idx=cfg.pad_token_id)\\n        self.embed_proj = (\\n            nn.Linear(cfg.token_embed_dim, cfg.d_model)\\n            if cfg.token_embed_dim != cfg.d_model\\n            else nn.Identity()\\n        )\\n        self.pos_embed = nn.Embedding(cfg.max_length, cfg.d_model)\\n        layer = nn.TransformerEncoderLayer(\\n            d_model=cfg.d_model,\\n            nhead=cfg.n_heads,\\n            dim_feedforward=cfg.mlp_dim,\\n            dropout=cfg.dropout,\\n            activation=\\"gelu\\",\\n            batch_first=True,\\n            norm_first=True,\\n        )\\n        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)\\n        self.norm = nn.LayerNorm(cfg.d_model)\\n        self.out = nn.Linear(cfg.d_model, cfg.output_dim)\\n\\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)\\n        hidden = self.embed_proj(self.token_embed(input_ids)) + self.pos_embed(positions)\\n        hidden = self.encoder(hidden, src_key_padding_mask=~attention_mask.bool())\\n        hidden = self.norm(hidden)\\n        return self.out(hidden)\\n\\n    def pooled(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\\n        token = self.forward(input_ids, attention_mask)\\n        mask = attention_mask.to(token.dtype).unsqueeze(-1)\\n        return (token * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)\\n", "src/slopdet/heads.py": "\\"\\"\\"Linear heads on a frozen student (or on a bag-of-features vector).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\ntry:\\n    import torch\\n    from torch import nn\\nexcept ImportError as exc:  # pragma: no cover\\n    raise ImportError(\\"heads.py needs torch. Install the train extra.\\") from exc\\n\\n\\n@dataclass\\nclass HeadConfig:\\n    input_dim: int\\n    n_lexical: int\\n    n_rhetorical: int\\n    n_jspace: int = 512\\n    n_construction: int = 5\\n    hidden: int = 128\\n\\n\\nclass Heads(nn.Module):\\n    def __init__(self, cfg: HeadConfig):\\n        super().__init__()\\n        self.cfg = cfg\\n        self.lexical = nn.Linear(cfg.input_dim, cfg.n_lexical)\\n        self.rhetorical = nn.Linear(cfg.input_dim, cfg.n_rhetorical)\\n        self.construction = nn.Linear(cfg.input_dim, cfg.n_construction)\\n        self.jspace = nn.Linear(cfg.input_dim, cfg.n_jspace)\\n        self.contrast = nn.Linear(cfg.input_dim, 1)\\n\\n    def forward(self, pooled: torch.Tensor) -> dict[str, torch.Tensor]:\\n        return {\\n            \\"lexical\\": self.lexical(pooled),\\n            \\"rhetorical\\": self.rhetorical(pooled),\\n            \\"construction\\": self.construction(pooled),\\n            \\"jspace\\": self.jspace(pooled),\\n            \\"contrast\\": self.contrast(pooled).squeeze(-1),\\n        }\\n", "src/slopdet/jlens.py": "\\"\\"\\"J-lens hook. Fit happens in the Colab notebook; this module is a placeholder.\\"\\"\\"\\n", "notebooks/colab_pipeline.py": "\\"\\"\\"Colab smoke/full pipeline. Imported by the notebook after the repo is written to disk.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport random\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.preprocessing import StandardScaler\\n\\nfrom slopdet.calibrate import calibration_record, human_percentile\\nfrom slopdet.construction import construction_stats\\nfrom slopdet.ontology import load_ontology\\nfrom slopdet.report import render_hits\\nfrom slopdet.weaklabel import label_text\\n\\nSEED_HUMAN = [\\n    \\"Thursday mornings at the clinic were empty. Half the early slots sat unused, and the monthly average hid it.\\",\\n    \\"I cut the section. The reader already had the date, the name, and the number.\\",\\n    \\"Maya sent the booking export with names stripped. I marked the gaps in a shared sheet.\\",\\n    \\"The RPC timed out at 3am. I almost scrapped the feature, then cached the last good payload.\\",\\n    \\"We shipped on Tuesday. Review time went from 30 minutes to 8.\\",\\n    \\"He said the eval was the product. I believed him after the third silent failure.\\",\\n    \\"OakNorth\'s 2019 report listed the default rate by vintage, not a slogan.\\",\\n    \\"Put the conclusion in the first sentence if the rest is method.\\",\\n    \\"I don\'t know if FineWeb is clean after 2022. Filter the dumps anyway.\\",\\n    \\"The student is 40 million parameters because the embedding table is huge.\\",\\n]\\n\\nSEED_MACHINE = [\\n    \\"Here\'s the thing, in today\'s competitive landscape we leverage robust pipelines to unlock the power of detection.\\",\\n    \\"It\'s worth noting that, at its core, this is a testament to seamless, cutting-edge architecture.\\",\\n    \\"Additionally, experts agree the launch marks a pivotal moment, highlighting the team\'s commitment to better workflows.\\",\\n    \\"In conclusion, we hope this helps. Please don\'t hesitate to reach out as we move the needle going forward.\\",\\n    \\"As an AI, I can say the platform serves as a centralized hub, fostering synergy across pain points.\\",\\n    \\"What if I told you it\'s not just about accuracy \\u2014 it\'s about delivering value and supercharging outcomes?\\",\\n    \\"Certainly, the solution is multifaceted and meticulously designed to streamline your operations.\\",\\n    \\"In this article, we\'ll delve into how our transformative paradigm shift empowers teams.\\",\\n    \\"The company, nestled in the heart of the city, boasts a rich cultural heritage and a vibrant tapestry of innovation.\\",\\n    \\"Overall, studies show that utilizing these tools will elevate your process, underscoring its significance.\\",\\n]\\n\\n\\ndef seed_docs() -> list[dict[str, Any]]:\\n    docs = []\\n    for i, text in enumerate(SEED_HUMAN):\\n        docs.append({\\"id\\": f\\"seed-human-{i}\\", \\"text\\": text, \\"source\\": \\"seed\\", \\"model\\": \\"human\\", \\"pile\\": 0})\\n    for i, text in enumerate(SEED_MACHINE):\\n        docs.append({\\"id\\": f\\"seed-ai-{i}\\", \\"text\\": text, \\"source\\": \\"seed\\", \\"model\\": \\"template\\", \\"pile\\": 1})\\n    return docs\\n\\n\\ndef try_hc3(n: int, rng: random.Random) -> list[dict[str, Any]]:\\n    try:\\n        from datasets import load_dataset\\n    except ImportError:\\n        print(\\"datasets not installed; using seed corpus only\\")\\n        return []\\n    configs = (\\"reddit_eli5\\", \\"finance\\", \\"open_qa\\", \\"medicine\\", \\"wiki_csai\\")\\n    docs: list[dict[str, Any]] = []\\n    for cfg in configs:\\n        try:\\n            ds = load_dataset(\\"Hello-SimpleAI/HC3\\", cfg, split=\\"train\\")\\n        except Exception as exc:\\n            print(f\\"HC3 {cfg} skipped: {exc}\\")\\n            continue\\n        rows = list(ds)\\n        rng.shuffle(rows)\\n        for row in rows:\\n            if len(docs) >= n:\\n                return docs\\n            human = row.get(\\"human_answers\\") or []\\n            machine = row.get(\\"chatgpt_answers\\") or []\\n            if human:\\n                docs.append(\\n                    {\\n                        \\"id\\": f\\"hc3-{cfg}-h-{len(docs)}\\",\\n                        \\"text\\": human[0],\\n                        \\"source\\": \\"hc3\\",\\n                        \\"model\\": \\"human\\",\\n                        \\"pile\\": 0,\\n                    }\\n                )\\n            if machine and len(docs) < n:\\n                docs.append(\\n                    {\\n                        \\"id\\": f\\"hc3-{cfg}-m-{len(docs)}\\",\\n                        \\"text\\": machine[0],\\n                        \\"source\\": \\"hc3\\",\\n                        \\"model\\": \\"chatgpt\\",\\n                        \\"pile\\": 1,\\n                    }\\n                )\\n        if docs:\\n            break\\n    return docs\\n\\n\\ndef featurize(doc: dict[str, Any], id_index: dict[str, int], dim: int) -> np.ndarray:\\n    vec = np.zeros(dim, dtype=np.float32)\\n    for hit in doc.get(\\"style_hits\\") or []:\\n        idx = id_index.get(hit[\\"id\\"])\\n        if idx is not None:\\n            vec[idx] += 1.0\\n    stats = doc.get(\\"construction\\") or {}\\n    extra = [\\n        float(stats.get(\\"burstiness\\") or 0.0),\\n        float(stats.get(\\"evenness\\") or 0.0),\\n        float(stats.get(\\"recap_closure\\") or 0.0),\\n        float(stats.get(\\"over_explain\\") or 0.0),\\n        float(stats.get(\\"portability\\") or 0.0),\\n        float(stats.get(\\"n_words\\") or 0.0) / 1000.0,\\n    ]\\n    vec = np.concatenate([vec, np.array(extra, dtype=np.float32)])\\n    return vec\\n\\n\\ndef build_and_train(root: Path, n_docs: int = 400) -> dict[str, Any]:\\n    rng = random.Random(0)\\n    onto = load_ontology(root / \\"ontology\\")\\n    ids = [p.id for p in onto.enabled_patterns()]\\n    id_index = {pid: i for i, pid in enumerate(ids)}\\n\\n    docs = seed_docs()\\n    hc3 = try_hc3(max(0, n_docs - len(docs)), rng)\\n    docs.extend(hc3)\\n    print(f\\"corpus: {len(docs)} docs (seed={len(SEED_HUMAN)+len(SEED_MACHINE)} hc3={len(hc3)})\\")\\n\\n    for doc in docs:\\n        text = doc[\\"text\\"]\\n        doc[\\"style_hits\\"] = [\\n            {k: h[k] for k in (\\"id\\", \\"start\\", \\"end\\", \\"unit\\")} for h in label_text(text, onto)\\n        ]\\n        doc[\\"construction\\"] = construction_stats(text)\\n        doc[\\"jspace\\"] = []\\n\\n    data_dir = root / \\"data\\"\\n    data_dir.mkdir(parents=True, exist_ok=True)\\n    corpus_path = data_dir / \\"corpus.jsonl\\"\\n    with corpus_path.open(\\"w\\", encoding=\\"utf-8\\") as fh:\\n        for doc in docs:\\n            fh.write(json.dumps(doc, ensure_ascii=False) + \\"\\\\n\\")\\n\\n    X = np.stack([featurize(d, id_index, len(ids)) for d in docs])\\n    y = np.array([d[\\"pile\\"] for d in docs], dtype=np.int64)\\n    scaler = StandardScaler()\\n    Xs = scaler.fit_transform(X)\\n    clf = LogisticRegression(max_iter=1000, class_weight=\\"balanced\\")\\n    clf.fit(Xs, y)\\n\\n    human_scores = clf.predict_proba(Xs[y == 0])[:, 1].tolist()\\n    calib = calibration_record(human_scores, 0.01)\\n    artifacts = root / \\"artifacts\\"\\n    artifacts.mkdir(parents=True, exist_ok=True)\\n    bundle = {\\n        \\"pattern_ids\\": ids,\\n        \\"coef\\": clf.coef_[0].tolist(),\\n        \\"intercept\\": float(clf.intercept_[0]),\\n        \\"scaler_mean\\": scaler.mean_.tolist(),\\n        \\"scaler_scale\\": scaler.scale_.tolist(),\\n        \\"calibration\\": calib,\\n        \\"ontology_sha256\\": onto.sha256,\\n        \\"n_docs\\": len(docs),\\n        \\"never_trained_on\\": [\\"raid-test\\"],\\n    }\\n    bundle_path = artifacts / \\"sklearn_bundle.json\\"\\n    bundle_path.write_text(json.dumps(bundle), encoding=\\"utf-8\\")\\n    print(\\"wrote\\", bundle_path, \\"threshold\\", calib[\\"threshold\\"])\\n    return {\\"docs\\": docs, \\"clf\\": clf, \\"scaler\\": scaler, \\"onto\\": onto, \\"bundle\\": bundle, \\"ids\\": ids}\\n\\n\\ndef score_text(text: str, *, onto, clf, scaler, ids, human_scores: list[float]) -> dict[str, Any]:\\n    id_index = {pid: i for i, pid in enumerate(ids)}\\n    hits = label_text(text, onto)\\n    doc = {\\"style_hits\\": hits, \\"construction\\": construction_stats(text)}\\n    x = scaler.transform([featurize(doc, id_index, len(ids))])\\n    score = float(clf.predict_proba(x)[0, 1])\\n    pct = human_percentile(score, human_scores)\\n    result = render_hits(hits, resemblance={\\"human_percentile\\": pct})\\n    result[\\"construction\\"] = doc[\\"construction\\"]\\n    result[\\"matches_ai_pile_score\\"] = score\\n    return result\\n\\n\\ndef demo(state: dict[str, Any]) -> None:\\n    docs = state[\\"docs\\"]\\n    human_scores = [\\n        float(state[\\"clf\\"].predict_proba(state[\\"scaler\\"].transform([\\n            featurize(d, {pid: i for i, pid in enumerate(state[\\"ids\\"])}, len(state[\\"ids\\"]))\\n        ]))[0, 1])\\n        for d in docs\\n        if d[\\"pile\\"] == 0\\n    ]\\n    samples = [\\n        \\"Here\'s the thing, in today\'s world we leverage robust tools. In conclusion, experts agree.\\",\\n        \\"Thursday mornings at the clinic were empty. I marked the gaps on a sheet.\\",\\n    ]\\n    for text in samples:\\n        print(\\"=\\" * 60)\\n        print(text)\\n        out = score_text(\\n            text,\\n            onto=state[\\"onto\\"],\\n            clf=state[\\"clf\\"],\\n            scaler=state[\\"scaler\\"],\\n            ids=state[\\"ids\\"],\\n            human_scores=human_scores,\\n        )\\n        print(out.get(\\"style_summary\\") or \\"\\")\\n        for hit in out[\\"hits\\"][:8]:\\n            print(f\\"  [{hit[\'id\']}] {hit[\'quote\']!r}\\")\\n            print(f\\"      {hit[\'fix\']}\\")\\n        if out.get(\\"resemblance\\") and len(human_scores) >= 30:\\n            print(out[\\"resemblance\\"][\\"text\\"])\\n        elif out.get(\\"resemblance\\"):\\n            print(\\"(resemblance hidden until \\u226530 human reference docs; HC3 will fill this on Colab)\\")\\n\\n\\ndef try_gpu_distill(root: Path, docs: list[dict[str, Any]], max_docs: int = 64) -> None:\\n    \\"\\"\\"Best-effort teacher cache + 1-epoch student. Never fails the notebook.\\"\\"\\"\\n    try:\\n        import torch\\n        from transformers import AutoModelForCausalLM, AutoTokenizer\\n        from slopdet.student import Student, StudentConfig\\n    except Exception as exc:\\n        print(\\"GPU distill skipped (imports):\\", exc)\\n        return\\n    if not torch.cuda.is_available():\\n        print(\\"GPU distill skipped: no CUDA\\")\\n        return\\n\\n    token = os.environ.get(\\"HF_TOKEN\\") or os.environ.get(\\"HUGGING_FACE_HUB_TOKEN\\")\\n    candidates = []\\n    if os.environ.get(\\"FULL\\") == \\"1\\":\\n        candidates.append(\\"google/gemma-3-4b-it\\")\\n    candidates.append(\\"Qwen/Qwen2.5-0.5B-Instruct\\")\\n\\n    model = None\\n    tokenizer = None\\n    name = None\\n    for cand in candidates:\\n        try:\\n            print(\\"loading teacher\\", cand)\\n            tokenizer = AutoTokenizer.from_pretrained(cand, token=token)\\n            model = AutoModelForCausalLM.from_pretrained(\\n                cand,\\n                token=token,\\n                torch_dtype=torch.float16,\\n                device_map=\\"auto\\",\\n            )\\n            name = cand\\n            break\\n        except Exception as exc:\\n            print(\\"teacher failed\\", cand, exc)\\n            model = None\\n    if model is None or tokenizer is None:\\n        print(\\"GPU distill skipped: no teacher\\")\\n        return\\n\\n    hidden = int(getattr(model.config, \\"hidden_size\\", 896))\\n    n_layers = int(getattr(model.config, \\"num_hidden_layers\\", 24))\\n    layer_idx = min(17, n_layers - 1)\\n    print(\\"teacher\\", name, \\"hidden\\", hidden, \\"L\\", layer_idx)\\n    if tokenizer.pad_token is None:\\n        tokenizer.pad_token = tokenizer.eos_token\\n\\n    subset = docs[:max_docs]\\n    device = torch.device(\\"cuda\\")\\n    cfg = StudentConfig(\\n        vocab_size=len(tokenizer),\\n        pad_token_id=int(tokenizer.pad_token_id or 0),\\n        max_length=128,\\n        output_dim=hidden,\\n        n_layers=2,\\n    )\\n    student = Student(cfg).to(device)\\n    opt = torch.optim.AdamW(student.parameters(), lr=3e-4)\\n    model.eval()\\n    student.train()\\n    losses = []\\n    for step, doc in enumerate(subset):\\n        enc = tokenizer(\\n            doc[\\"text\\"],\\n            return_tensors=\\"pt\\",\\n            truncation=True,\\n            max_length=128,\\n            padding=\\"max_length\\",\\n        )\\n        enc = {k: v.to(device) for k, v in enc.items()}\\n        with torch.no_grad():\\n            out = model(**enc, output_hidden_states=True)\\n            target = out.hidden_states[layer_idx].float()\\n        pred = student(enc[\\"input_ids\\"], enc[\\"attention_mask\\"]).float()\\n        mask = enc[\\"attention_mask\\"].unsqueeze(-1).float()\\n        loss = ((pred - target) ** 2 * mask).sum() / mask.sum().clamp_min(1.0)\\n        opt.zero_grad()\\n        loss.backward()\\n        opt.step()\\n        losses.append(float(loss.detach()))\\n        if step % 16 == 0:\\n            print(f\\"distill {step}/{len(subset)} loss={losses[-1]:.4f}\\")\\n    artifacts = root / \\"artifacts\\"\\n    artifacts.mkdir(parents=True, exist_ok=True)\\n    ckpt = artifacts / \\"student_smoke.pt\\"\\n    torch.save({\\"cfg\\": cfg.__dict__, \\"state\\": student.state_dict(), \\"teacher\\": name, \\"hidden\\": hidden}, ckpt)\\n    print(\\"saved\\", ckpt, \\"mean loss\\", sum(losses) / max(len(losses), 1))\\n    del model\\n    torch.cuda.empty_cache()\\n"}')
for rel, content in FILES.items():
    path = Path(rel)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
print("wrote", len(FILES), "files")
import sys
sys.path.insert(0, str(Path("src").resolve()))
print("ITAIS ready")


In [ ]:
# Train smoke detector (seed + HC3 if downloadable)
from pathlib import Path
import sys
sys.path.insert(0, str(Path("src").resolve()))
sys.path.insert(0, str(Path("notebooks").resolve()))
from colab_pipeline import build_and_train, demo, try_gpu_distill

state = build_and_train(Path(".").resolve(), n_docs=N_DOCS)
demo(state)


In [ ]:
# Optional GPU distill (skipped automatically on CPU / failed teacher load)
from pathlib import Path
try_gpu_distill(Path(".").resolve(), state["docs"], max_docs=32 if SMOKE else 256)
print("done")
